# Generate BibTeX from DOI List

Paste a list of DOIs below, then run all cells to fetch BibTeX entries and write them to a `.bib` file. DOI URLs such as `https://doi.org/...` and plain DOI strings such as `10.1000/example` are both accepted.

The notebook first uses DOI content negotiation through `doi.org`, then falls back to Crossref's BibTeX transform endpoint when needed. Saved entries are formatted with one BibTeX field per line.

In [1]:
# ---------- User inputs ----------

doi_list = [
    "10.1038/nphys1170",
    # "https://doi.org/10.1145/3375630",
]

output_bib_file = "references_from_dois.bib"

# Set to True if you want to append to output_bib_file instead of replacing it.
append_to_existing_file = False

# Optional but polite for DOI/Crossref requests. Example: "name@university.edu"
contact_email = ""

request_timeout_seconds = 30
request_pause_seconds = 0.2

In [2]:
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

In [3]:
def normalize_doi(value):
    """Normalize a plain DOI or DOI URL into just the DOI string."""
    doi = str(value).strip().strip("<>")
    doi = doi.replace("\u200b", "").strip()
    doi = re.sub(r"^https?://(dx\.)?doi\.org/", "", doi, flags=re.IGNORECASE)
    doi = re.sub(r"^doi:\s*", "", doi, flags=re.IGNORECASE)
    doi = urllib.parse.unquote(doi)
    return doi.strip().rstrip(".,;")


def build_user_agent(contact_email=""):
    user_agent = "pub-assist-doi-to-bibtex/1.0"
    if contact_email:
        user_agent += f" (mailto:{contact_email})"
    return user_agent


def read_url_text(url, headers, timeout):
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=timeout) as response:
        charset = response.headers.get_content_charset() or "utf-8"
        return response.read().decode(charset, errors="replace")


def fetch_bibtex_from_doi_org(doi, timeout, contact_email=""):
    encoded_doi = urllib.parse.quote(doi, safe="/")
    url = f"https://doi.org/{encoded_doi}"
    headers = {
        "Accept": "application/x-bibtex",
        "User-Agent": build_user_agent(contact_email),
    }
    return read_url_text(url, headers, timeout)


def fetch_bibtex_from_crossref(doi, timeout, contact_email=""):
    encoded_doi = urllib.parse.quote(doi, safe="")
    url = f"https://api.crossref.org/works/{encoded_doi}/transform/application/x-bibtex"
    headers = {
        "Accept": "application/x-bibtex",
        "User-Agent": build_user_agent(contact_email),
    }
    return read_url_text(url, headers, timeout)

In [4]:
def format_bibtex_entry(bibtex, indent="  "):
    """Put each top-level BibTeX field on its own line."""
    text = " ".join(str(bibtex).strip().split())
    if not text.startswith("@"):
        return str(bibtex).strip()

    def format_segment(segment):
        segment = segment.strip()
        if segment.startswith("@") or segment == "}":
            return segment
        return re.sub(
            r"^([^=]+?)\s*=\s*",
            lambda match: f"{match.group(1).strip()} = ",
            segment,
            count=1,
        )

    lines = []
    current = []
    brace_depth = 0
    in_quote = False
    escaped = False

    def append_line(segment):
        segment = format_segment(segment)
        if segment:
            lines.append(segment if not lines else indent + segment)

    for char in text:
        current.append(char)

        if escaped:
            escaped = False
            continue
        if char == "\\":
            escaped = True
            continue
        if char == '"' and brace_depth == 1:
            in_quote = not in_quote
            continue
        if in_quote:
            continue
        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1
            if brace_depth == 0:
                final_segment = "".join(current).rstrip()
                append_line(final_segment[:-1] if final_segment.endswith("}") else final_segment)
                lines.append("}")
                current = []
        elif char == "," and brace_depth == 1:
            append_line("".join(current))
            current = []

    trailing = "".join(current).strip()
    if trailing:
        append_line(trailing)

    return "\n".join(line.rstrip() for line in lines if line.strip())


def fetch_bibtex(doi, timeout=30, contact_email=""):
    """Fetch one BibTeX entry, trying doi.org first and Crossref second."""
    errors = []

    for source_name, fetcher in [
        ("doi.org", fetch_bibtex_from_doi_org),
        ("crossref", fetch_bibtex_from_crossref),
    ]:
        try:
            bibtex = fetcher(doi, timeout=timeout, contact_email=contact_email).strip()
            if bibtex.startswith("@"):
                return format_bibtex_entry(bibtex)
            errors.append(f"{source_name}: response was not BibTeX")
        except urllib.error.HTTPError as exc:
            errors.append(f"{source_name}: HTTP {exc.code} {exc.reason}")
        except Exception as exc:
            errors.append(f"{source_name}: {exc}")

    raise RuntimeError("; ".join(errors))


def generate_bibtex_entries(dois, timeout=30, contact_email="", pause_seconds=0.2):
    entries = []
    failures = []
    seen = set()

    for raw_doi in dois:
        doi = normalize_doi(raw_doi)
        if not doi:
            continue

        doi_key = doi.lower()
        if doi_key in seen:
            print(f"Skipping duplicate DOI: {doi}")
            continue
        seen.add(doi_key)

        try:
            print(f"Fetching BibTeX for {doi} ...")
            entries.append((doi, fetch_bibtex(doi, timeout=timeout, contact_email=contact_email)))
        except Exception as exc:
            failures.append((doi, str(exc)))

        time.sleep(pause_seconds)

    return entries, failures


def write_bibtex_file(entries, output_file, append=False):
    output_path = Path(output_file)
    mode = "a" if append and output_path.exists() else "w"

    with output_path.open(mode, encoding="utf-8") as file:
        if mode == "a" and output_path.stat().st_size > 0:
            file.write("\n\n")
        file.write("\n\n".join(bibtex for _, bibtex in entries))
        if entries:
            file.write("\n")

    return output_path.resolve()

In [5]:
entries, failures = generate_bibtex_entries(
    doi_list,
    timeout=request_timeout_seconds,
    contact_email=contact_email,
    pause_seconds=request_pause_seconds,
)

output_path = write_bibtex_file(
    entries,
    output_bib_file,
    append=append_to_existing_file,
)

print(f"\nWrote {len(entries)} BibTeX entr{'y' if len(entries) == 1 else 'ies'} to: {output_path}")

if failures:
    print("\nFailed DOI lookups:")
    for doi, error in failures:
        print(f"- {doi}: {error}")

Fetching BibTeX for 10.1038/nphys1170 ...

Wrote 1 BibTeX entry to: E:\github_codes\pub_assist\pub_assist\references_from_dois.bib


In [6]:
# Optional: preview generated BibTeX entries in the notebook.
for doi, bibtex in entries:
    print(f"% DOI: {doi}")
    print(bibtex)
    print()

% DOI: 10.1038/nphys1170
@article{Aspelmeyer_2009,
  title = {Measured measurement},
  volume = {5},
  ISSN = {1745-2481},
  url = {http://dx.doi.org/10.1038/nphys1170},
  DOI = {10.1038/nphys1170},
  number = {1},
  journal = {Nature Physics},
  publisher = {Springer Science and Business Media LLC},
  author = {Aspelmeyer, Markus},
  year = {2009},
  month = Jan,
  pages = {11–12}
}

